# Part 4: Using GPU acceleration with PyTorch

## Manual use of `.cuda()`

Now the magic of PyTorch comes in. So far, we've only been using the CPU to do computation. When we want to scale to a bigger problem, that won't be feasible for very long.
PyTorch makes it really easy to use the GPU for accelerating computation. Consider the following code that computes the element-wise product of two large matrices:

In [1]:
import torch

t1 = torch.randn(1000, 1000)
t2 = torch.randn(1000, 1000)
t3 = t1*t2
print(t3)

tensor([[-0.0393,  0.1842,  0.0448,  ..., -0.0073, -0.2732,  0.1029],
        [-0.4855, -0.3304,  0.3108,  ...,  1.3413, -0.6314,  2.2966],
        [-0.6738, -0.4335,  1.7299,  ...,  1.0226,  0.8955, -0.3654],
        ...,
        [-0.5833,  0.7521,  1.6441,  ..., -0.9688, -1.1111, -0.1056],
        [ 0.1055,  0.0069, -0.2708,  ...,  0.8502, -0.1010, -0.9782],
        [ 0.0051, -0.1013,  0.0539,  ...,  0.4206,  0.8081, -1.3488]])


The convenience method `.cuda()` moves a tensor to a CUDA GPU, but it fails on machines without CUDA. The following cell therefore only runs the CUDA example when a compatible GPU is available:

In [2]:

if torch.cuda.is_available():
    t1 = torch.randn(1000, 1000).cuda()
    t2 = torch.randn(1000, 1000).cuda()
    t3 = t1 * t2
    print(t3)
else:
    print("CUDA is not available; enable a GPU runtime in Colab to run this example.")

tensor([[-1.1266e+00,  1.1092e+00, -1.4255e+00,  ..., -5.1948e-01,
         -6.2302e-01, -2.0239e-01],
        [ 4.7352e-01,  1.3888e+00, -4.2473e-01,  ...,  8.9331e-01,
          1.7832e+00,  5.6211e-01],
        [ 3.8555e-02,  1.8644e-01, -1.0161e-01,  ...,  9.8644e-01,
          3.6081e+00,  6.5289e-01],
        ...,
        [ 1.2017e+00, -7.4648e-02,  5.7320e-01,  ...,  9.2466e-01,
          3.3159e+00,  6.0796e-01],
        [-3.0627e-01,  4.8729e-01,  2.6162e-01,  ...,  2.3085e+00,
         -8.4754e-01,  2.1802e+00],
        [ 1.2951e+00, -7.0734e-02, -5.0242e-02,  ..., -2.0129e-03,
         -5.5048e-02,  3.5351e-01]], device='cuda:0')


If you're running this notebook in colab, now enable GPU acceleration (`Runtime->Runtime Type` and add a `GPU` in the hardware accelerator pull-down). You'll then need to re-run all cells to this point.

If you were able to run the above with hardware acceleration, the print-out of the result tensor would show that it was an instance of `cuda.FloatTensor` type on the `(GPU 0)` GPU device. If you wanted to copy the tensor back to the CPU, you would use the `.cpu()` method.

## Writing platform agnostic code

Most of the time you'd like to write code that is device agnostic; that is it will run on a GPU if one is available, and otherwise it would fall back to the CPU. The recommended way to do this is as follows:

In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
t1 = torch.randn(1000, 1000).to(device)
t2 = torch.randn(1000, 1000).to(device)
t3 = t1*t2
print(t3)

tensor([[ 2.7002, -0.2477,  0.9965,  ...,  1.6105,  0.2763, -0.9244],
        [-0.2866,  5.6205,  0.0647,  ..., -0.4455,  0.6633,  0.5410],
        [-1.0051, -1.4874, -2.0104,  ...,  0.4020,  0.8270, -0.0239],
        ...,
        [-0.9911,  0.1165,  0.6709,  ..., -0.1377, -0.9081,  0.2485],
        [-0.3546,  0.0329,  0.1368,  ..., -0.0164,  0.0670, -0.7246],
        [ 0.4928, -1.4540, -0.9684,  ..., -0.6699, -0.2752,  0.3522]],
       device='cuda:0')


## Accelerating neural net training

If you wanted to accelerate the training of a neural net using raw PyTorch, you would have to copy both the model and the training data to the GPU. Unless you were using a really small dataset like MNIST, you would typically _stream_ the batches of training data to the GPU as you used them in the training loop:

```python
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = BaselineModel(784, 784, 10).to(device)

loss_function = ...
optimiser = ...

for epoch in range(10):
    for data in trainloader:
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimiser.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimiser.step()
```

Using Torchbearer, this becomes much simpler - you just tell the `Trial` to run on the GPU and that's it!:

```python
model = BetterCNN()

loss_function = ...
optimiser = ...

device = "cuda:0" if torch.cuda.is_available() else "cpu"
trial = Trial(model, optimiser, loss_function, metrics=['loss', 'accuracy']).to(device)
trial.with_generators(trainloader)
trial.run(epochs=10)
```


## Multiple GPUs

Using multiple GPUs is beyond the scope of the lab, but if you have multiple cuda devices, they can be referred to by index: `cuda:0`, `cuda:1`, `cuda:2`, etc. You have to be careful not to mix operations on different devices, and would need to carefully orchestrate moving of data between the devices (which can really slow down your code to the point at which using the CPU would actually be faster).

## Questions

__Answer the following questions (enter the answer in the box below each one):__

__1.__ What features of GPUs allow them to perform computations faster than a typical CPU?

GPUs have thousands of simple cores, where a CPU has a handful of complex ones, so they can do a huge number of arithmetic operations in parallel. That fits deep learning well, because training is dominated by matrix multiplications and element-wise operations that are the same calculation repeated across large arrays of independent values.
GPUs also have much higher memory bandwidth than CPUs, so data reaches those cores fast enough to keep them busy. Many recent GPUs add specialised hardware such as tensor cores, which speed up matrix maths further at lower precision.

__2.__ What is the biggest limiting factor for training large models with current generation GPUs?

The biggest limit is the amount of memory on the GPU. During training it has to hold more than just the model’s weights. It also stores the gradients, the optimiser’s extra values (Adam keeps two for every parameter), and the activations from the forward pass that backprop needs. Even good GPUs only have a few tens of GB, so a large model or a big batch size can run out of memory quickly. You then have to work around it, for example by using smaller batches, lower precision, or splitting the model across several GPUs. Splitting has its own cost, because moving data between GPUs is slow, as the notebook mentions.